## Initialization

### Imports

In [ ]:
# Importing needed code

from typing import (
    Callable,
    Literal
)
from pathlib import Path
from random import sample

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.interpolate import make_interp_spline

from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing import helpers
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    EnergyColumn,
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.processing.neutron_window_strategy.abstract_strategy \
    import AbstractNeutronStrategy


### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> proc_types.NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = proc_types.NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


In [ ]:
def relative_rmse(values: list[tuple[pd.Series | float, pd.Series | float]]) -> pd.Series | float:
    rel_sq_values = [relative_square_error(x, x_err) for x, x_err in values]
    rel_sq_sum = sum(rel_sq_values)
    if isinstance(rel_sq_sum, pd.Series):
        return rel_sq_sum.pow(1./2)
    else:
        return rel_sq_sum ** (1./2)


def relative_square_error(x: pd.Series | float, x_err: pd.Series | float) -> pd.Series | float:
    # divide x_err by x
    # square it
    # return
    rel_err = x_err / x
    if isinstance(rel_err, pd.Series):
        return rel_err.pow(2).fillna(0)
    else:
        return rel_err ** 2

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pd.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    
    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)
    
    signals_np = -signals_np + baselines + offset
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def integrate_pulses(pulse_data: np.array, t_start: int, t_end: int):
    left_vals = pulse_data[:, t_start : t_end]
    right_vals = pulse_data[:, t_start+1 : t_end+1]
    
    print(left_vals.shape, right_vals.shape)
    print(left_vals)
    print(right_vals)
    midpoints = (left_vals + right_vals) / 2
    print(midpoints)
    column_areas = midpoints * 2  ## 2 ns between data points
    print(column_areas)
    areas = column_areas.sum(axis=1)
    return areas

## Data Loading

### Loading Params

In [ ]:
background_id = "TB-46"
reactor_id = "TB-26"
base_experiment_ids = [background_id, reactor_id]
voltages = [1250, 1300, 1350, 1400, 1450, 1475, 1500, 1525, 1550]
voltage_ids = [f"TB-bias-{voltage}" for voltage in voltages]
experiment_ids = [*base_experiment_ids, *voltage_ids]

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

### Loading and Initial Processing

In [ ]:
figure_data = {k: {} for k in ["a", "b", "de"]}

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Ensure that index matches between signals and CAEN data
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    signals_df = exp_data["signals_df"]
    
    unclassified_index: pd.Index = unclassified_df.index
    signals_index: pd.Index = signals_df.index
    clean_index = unclassified_index.intersection(signals_index)
    
    unclassified_df = unclassified_df.loc[clean_index]
    signals_df = signals_df.loc[clean_index]
    
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df
    exp_data["signals_df"] = signals_df

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Neutron Classification

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = proc.get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    df, df_err = proc.scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style="peak_finder",
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
    borders = exp_data[ExperimentDataKey.BORDERS]

    psd_report = proc.classify(
        psd_report,
        calibrated_energy_column,
        borders,
        DetectorDataframeColumn.NEW_N_CLASS
    )

    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Pulse Processing

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    signals_df = exp_data["signals_df"].astype("int32")
    
    signals_df = correct_raw_signals(signals_df)
    heights = signals_df.max(axis=1)
    signals_df.columns = signals_df.columns.map(int)
    
    psd_report["peak_height"] = heights
    exp_data["signals_df"] = signals_df
    exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Neutron/Gamma Separation

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
    n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value
    
    gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
    neutrons_only = psd_report.query(n_class_col_name).copy()
    exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
    exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    
    neutron_signals = signals_df.loc[neutrons_only.index]
    gamma_signals = signals_df.loc[gamma_only.index]

    exp_data["neutron_signals"] = neutron_signals
    exp_data["gamma_signals"] = gamma_signals

### Figure 2a Processing

In [ ]:
exp_id = "TB-46"

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["a"]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
peak_height = neutrons_only["peak_height"]

energy_bins = np.arange(0, 12000, step=200)

Z_n, *_ = np.histogram(peak_height, bins=energy_bins)
exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
    "neutron": {"standard": Z_n, "bins": energy_bins},
}
plot_data["histogram"] = {"counts": Z_n, "bins": energy_bins}

In [ ]:
# moving average
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["a"]
phd_histogram_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
phd_n_histogram = phd_histogram_data["neutron"]["standard"]

phd_n_moving_average = moving_average_centered(phd_n_histogram)

phd_histogram_data["neutron"]["moving_average"] = phd_n_moving_average
plot_data["histogram"]["moving_average"] = phd_n_moving_average

In [ ]:
exp_data = experiment_neutron_data[exp_id]
plot_data = figure_data["a"]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
phd_histogram_data = exp_data[
    ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["neutron"]
n_signals_df = exp_data["neutron_signals"]
energy_bins = phd_histogram_data["bins"]

energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
trace_bin_lo = energy_bins[::5][1:]
trace_bin_hi = energy_bins[1::5][1:]
trace_bins = list(zip(trace_bin_lo, trace_bin_hi))
traces = []
for trace_bin in trace_bins:
    bin_lo, bin_hi = trace_bin
    bin_mid = (bin_lo + bin_hi) / 2
    matching_neutrons = neutrons_only[neutrons_only["peak_height"].between(bin_lo, bin_hi)]
    if len(matching_neutrons) == 0:
        continue
    for neutron_id in matching_neutrons.index:
        matching_trace = n_signals_df.loc[neutron_id]
        
        peaks, peak_data = find_peaks(matching_trace, height=200, prominence=50)
        filtered_peaks = [peak for peak in peaks if abs(peak - 50) > 15]
        if len(filtered_peaks) == 0:
            traces.append((bin_mid, matching_trace))
            break
traces = sample(traces, len(traces))

plot_data["selected_traces"] = traces

### Figure 2b Processing

In [ ]:
figure_2b_exp_ids = [
    exp_id for exp_id in experiment_ids
    if "bias" in exp_id
]
figure_2b_plot_data = {exp_id: {} for exp_id in figure_2b_exp_ids}

In [ ]:
signals_count = 1500000
bin_width = 100
for exp_id in figure_2b_exp_ids:
    exp_data = experiment_neutron_data[exp_id]
    particles_subset = exp_data[ExperimentDataKey.PSD_REPORT]\
        .iloc[:signals_count, :].copy()
    signals_subset = exp_data["signals_df"]\
        .loc[particles_subset.index]
    
    sub_peak_height = particles_subset["peak_height"]

    bins = np.arange(0, 15000, step=bin_width)
    Zh_sub, *_ = np.histogram(sub_peak_height, bins=bins)
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
        "counts": Zh_sub,
        # "n_counts": Zh_n,
        "bins": bins
    }

In [ ]:
for exp_id in figure_2b_exp_ids:
    exp_data = experiment_neutron_data[exp_id]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    phd_counts = phd_data["counts"]

    norm_factor = 1 / phd_counts.max()
    phd_data["norm_factor"] = norm_factor

In [ ]:
for exp_id in figure_2b_exp_ids:
    exp_data = experiment_neutron_data[exp_id]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    figure_2b_plot_data[exp_id] = {**phd_data}
figure_data["b"] = figure_2b_plot_data

### Figure 2d/e Processing

In [ ]:
exp_ids = base_experiment_ids

In [ ]:
bin_start = 0
bin_end = 16000
bin_width = 50
bins = np.arange(bin_start, bin_end+bin_width, step=bin_width)

for exp_id in base_experiment_ids:
    exp_data = experiment_neutron_data[exp_id]
    plot_data = figure_data["de"]
    neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    all_particles = exp_data[ExperimentDataKey.PSD_REPORT]
    n_peak_height = neutrons_only["peak_height"]
    peak_height = all_particles["peak_height"]
    
    Z, *_ = np.histogram(peak_height, bins=bins)
    Zn, *_ = np.histogram(n_peak_height, bins=bins)
    
    figure_plot_data = {
        "counts": Z,
        "n_counts": Zn,
        "bins": bins
    }
    plot_data[exp_id] = figure_plot_data

In [ ]:
exp_data = experiment_neutron_data[background_id]
plot_data = figure_data["de"][background_id]
Z = plot_data["counts"]
bins = plot_data["bins"]

bin_mids = (bins[1:] + bins[:-1]) / 2

bin_above_lo = bin_mids >= 11000
bin_below_hi = bin_mids <= 15500
bin_mask = bin_above_lo & bin_below_hi

masked_bins = bin_mids[bin_mask]
masked_Z = Z[bin_mask]

fit_params, *_ = curve_fit(
    proc.gaussian, masked_bins, masked_Z,
    p0=(12000, 1000, 10000)
)
plot_data["fit_params"] = fit_params

## Figure Base Data

In [ ]:
dl_folder = Path.home() / "Downloads" / "neutron_detection_paper"
dl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_figure_2a_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Callable]:
    """
    First DataFrame holds neutron light output trace data from selected signals
    Each trace was selected from a different light energy bin
    The column labels are the midpoint of that trace's bin
    Second DataFrame contains histogram of pulse heights from neutron detector data
    Each row represents a single bin, and the counts, bin midpoint and bin width are provided
    Third DataFrame contains the histogram data in curve form
    The index is the bin midpoint, and the "y" series is the count
    Last return value is an interpolator function for the histogram curve
    This interpolator can be given a pulse height (in ADC channels)
    It will return the interpolated histogram count for that pulse height
    """
    fig_data = figure_data["a"]
    hist_data = fig_data["histogram"]
    selected_traces = fig_data["selected_traces"]
    energy_bins = hist_data["bins"]
    phd_moving_avg = hist_data["moving_average"]
    phd_moving_avg = np.nan_to_num(phd_moving_avg)

    energy_bin_mids = (energy_bins[1:] + energy_bins[:-1]) / 2
    energy_bin_widths = energy_bins[1:] - energy_bins[:-1]

    phd_interp = make_interp_spline(
        energy_bin_mids, phd_moving_avg,
        k=1
    )
    phd_interp_x = np.linspace(0, energy_bins.max(), num=1000)
    phd_interp_y = phd_interp(phd_interp_x)
    
    trace_df = pd.DataFrame(
        data={bin_mid: trace for bin_mid, trace in selected_traces}
    )
    phd_curve_df = pd.DataFrame(
        data={"y": phd_interp_y},
        index=phd_interp_x
    )
    phd_df = pd.DataFrame(
        data={
            "energy_bin_mids": energy_bin_mids,
            "energy_bin_widths": energy_bin_widths,
            "counts": phd_moving_avg
        },
    )
    return trace_df, phd_df, phd_curve_df, phd_interp

In [ ]:
def get_figure_2b_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    This data relates to the effect of bias voltage on neutron detection
    First DataFrame contains histograms for each bias voltage trial
    Each series is from a different voltage trial, and the name is the bias voltage used
    The index is the pulse height, and the series values are the bin counts
    Second DataFrame gives a normalization factor for each trial
    This factor ensures that each histogram's maximum has the same y value
    Third DataFrame gives annotation positions for each trial's plot
    The index is the trial voltage, and the coordinates are given in the series
    """
    annot_positions = {
        "1200": (1300, 1.0),
        "1250": (1350, 0.95),
        "1300": (1950, 0.59),
        "1350": (2750, 0.36),
        "1400": (4230, 0.216),
        "1450": (6350, 0.135),
        "1475": (7775, 0.109),
        "1500": (9425, 0.1),
        "1525": (11400, 0.08),
        "1550": (13850, 0.075)
    }

    fig_data = figure_data["b"]
    for k, v in fig_data.items():
        bins = v["bins"]
        v["bin_mids"] = (bins[1:] + bins[:-1]) / 2
    fig_series = [
        pd.Series(data=v["counts"], index=v["bin_mids"], name=k[-4:])
        for k, v in fig_data.items()
    ]
    fig_df = pd.concat(fig_series, axis=1)
    norm_factor_df = pd.DataFrame(
        data={k[-4:]: v["norm_factor"] for k, v in fig_data.items()},
        index=[0]
    )
    annot_positions_df = pd.DataFrame(
        data={
            "x": [x for x, _ in annot_positions.values()],
            "y": [y for _, y in annot_positions.values()]
        },
        index=annot_positions.keys()
    )
    return fig_df, norm_factor_df, annot_positions_df

In [ ]:
def get_figure_2d_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    First DataFrame contains histogram of neutron detection data
    The index is the pulse energy, and the series values are the bin counts
    Second DataFrame contains data for a fitted gaussian curve
    The index is the x values, and the series is the y values
    Third DataFrame contains fit parameters for the fitted gaussian curve
    The index labels each fit value, and their values are in the series
    """
    plot_data = figure_data["de"][background_id]
    counts = plot_data["counts"]
    bins = plot_data["bins"]
    fit_params = plot_data["fit_params"]

    bin_mids = (bins[1:] + bins[:-1]) / 2
    mu, sigma, _ = fit_params
    gaussian_x = np.linspace(mu - 3 * sigma, mu + 3 * sigma, 300)
    gaussian_y = proc.gaussian(gaussian_x, *fit_params)

    histo_df = pd.DataFrame(
        data={"counts": counts},
        index=bin_mids
    )
    gauss_df = pd.DataFrame(
        data={"y": gaussian_y},
        index=gaussian_x
    )
    fit_df = pd.DataFrame(
        data={"value": fit_params},
        index=["mu", "sigma", "A"]
    )
    return histo_df, gauss_df, fit_df

In [ ]:
def get_figure_2e_data() -> pd.DataFrame:
    """
    Contains histogram of detector data from background and neutron source experiments
    The index is the bin midpoint, and the values are the counts
    For the "background" series, all counts (i.e. gamma ray and neutrons) are included
    For the "neutron_source" series, only neutron counts are included
    """
    figure_2e_data = figure_data["de"]
    for k, v in figure_2e_data.items():
        bins = v["bins"]
        v["bin_mids"] = (bins[1:] + bins[:-1]) / 2
    fig_series = [
        pd.Series(
            data=v["counts" if k == background_id else "n_counts"],
            index=v["bin_mids"],
            name="background" if k == background_id else "neutron_source"
        )
        for k, v in figure_2e_data.items()]
    fig_df = pd.concat(fig_series, axis=1)
    return fig_df

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

### Plot Functions

In [ ]:
def plot_figure_2a(ax1: mpl.axes.Axes, ax2: mpl.axes.Axes, fig: mpl.figure.Figure):
    trace_df, phd_df, phd_curve_df, phd_interp = get_figure_2a_data()

    ax1.plot(phd_curve_df["y"], phd_curve_df.index, color="black")
    ax1.barh(
        phd_df["energy_bin_mids"], phd_df["counts"],
        height=phd_df["energy_bin_widths"],
        align="center",
        lw=3,
        ec=bg_blue,
        fc="#00000000"
    )
    for i, (bin_mid, trace) in enumerate(trace_df.items()):
        trace_x = [x + i * 25 for x in range(len(trace))]
        ax2.plot(trace_x, trace, label=bin_mid, lw=0)
        ax2.fill_between(
            trace_x, 0, trace, color=bg_blue, alpha=0.5
        )

    ax1.xaxis.set_inverted(True)
    ax1.xaxis.set_tick_params(labelbottom=False, bottom=False)
    ax1.yaxis.set_tick_params(labelbottom=False, bottom=False)
    ax2.xaxis.set_tick_params(labelbottom=False, bottom=False)
    ax2.yaxis.set_tick_params(labelbottom=False, direction="in")

    ax1.spines["top"].set_visible(False)
    ax1.spines["left"].set_visible(False)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    ax1.set_xlabel("Counts", fontsize=fontsize)
    ax2.set_ylabel(
        "Pulse height (ADC channel)", fontsize=fontsize
    )
    ax2.set_xlabel("Time (ns)", fontsize=fontsize)
    ax2.yaxis.set_label_position("left")
    ax2.yaxis.set_label_coords(0.075, 0.5)

    limits = (0, 5000)
    ax1.set_ylim(*limits)
    ax2.set_ylim(*limits)
    ax1.set_xlim(None, 5)
    ax2.set_xlim(25, 150)

    for i, (bin_mid, trace) in enumerate(trace_df.items()):
        bin_mid_idx = np.where(energy_bin_mids == bin_mid)[0]
        bin_lo = float(energy_bins[bin_mid_idx][0])
        bin_hi = float(energy_bins[bin_mid_idx+1][0])
        bin_phd_x = np.linspace(bin_lo, bin_hi).reshape(-1, 1)
        # bin_phd_y = phd_spline(bin_phd_x).reshape(-1, 1)
        bin_phd_y = phd_interp(bin_phd_x).reshape(-1, 1)
        bin_phd_xy = np.concatenate((bin_phd_y, bin_phd_x), axis=1)
        trace_max_x = float(trace.idxmax()) + (i * 25)
        bin_trace_xy = [
            [trace_max_x, bin_hi], [trace_max_x, bin_lo]
        ]

        ax1_to_display = ax1.transData.transform
        ax2_to_display = ax2.transData.transform
        display_to_figure = fig.transFigure.inverted().transform

        bin_phd_xy = display_to_figure(ax1_to_display(bin_phd_xy))
        bin_trace_xy = display_to_figure(
            ax2_to_display(bin_trace_xy)
        )
        bin_xy = np.concatenate((bin_phd_xy, bin_trace_xy))

        poly = mpl.patches.Polygon(
            bin_xy,
            closed=True,
            color=bg_bluegrey,
            alpha=0.3
        )
        fig.add_artist(poly)
    pass  # STUB

In [ ]:
def plot_figure_2b(ax: mpl.axes.Axes):
    fig_df, norm_factor_df, annot_positions_df = get_figure_2b_data()
    for voltage, plot_data in fig_df.items():
        annot_pos = annot_positions_df.loc[voltage]
        color = bg_blue if voltage == "1525" else bg_grey
        alpha = 0.3 if voltage == "1525" else 0.4

        counts = plot_data
        bin_mids = plot_data.index
        norm_factor = norm_factor_df.loc[0, voltage]

        ax.fill_between(
            bin_mids, counts * norm_factor,
            color=color,
            alpha=alpha
        )
        ax.annotate(
            f"{voltage} V", annot_pos,
            ha="center",
            va="baseline",
            fontsize=fontsize-2
        )

        ax.set_xlabel("Pulse height (ADC channel)", fontsize=fontsize)
        ax.set_ylabel("Normalized counts", fontsize=fontsize)
        ax.tick_params(labelsize=fontsize)

In [ ]:
def plot_figure_2d(ax: mpl.axes.Axes):
    color = bg_blue
    annot_x_offset = 40
    annot_y_offset = 200
    y_max = 20000

    histo_df, gauss_df, fit_df = get_figure_2d_data()
    mu = fit_df.loc["mu", "value"]
    sigma = fit_df.loc["sigma", "value"]

    ax.fill_between(histo_df.index, histo_df["counts"], color=color, alpha=0.5)

    ax.plot(gauss_df.index, gauss_df["y"], lw=2, color="black")

    ax.vlines([mu, mu+sigma], 0, y_max, lw=2, ls=":", color=bg_red)
    ax.annotate(
        r"$\mu$",
        (mu+annot_x_offset, y_max-annot_y_offset),
        ha="left",
        va="top",
        fontsize=fontsize-2
    )
    ax.annotate(
        r"$\mu+\sigma$",
        (mu+sigma+annot_x_offset, y_max-annot_y_offset),
        ha="left",
        va="top",
        fontsize=fontsize-2
    )

    ax.set_ylim(0, y_max)
    ax.set_xlabel("Pulse energy (ADC channel x 1000)", fontsize=fontsize)
    ax.set_ylabel("Counts (x 1000)", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)

In [ ]:
def plot_figure_2e(ax: mpl.axes.Axes):
    fig_df = get_figure_2e_data()
    
    for name, series in fig_df.items():
        color = bg_blue if name == "background" else bg_red

        ax.fill_between(series.index, series, color=color, alpha=0.5)

    ax.set_ylim(0, 35000)
    ax.set_xlim(0, 18000)
    ax.set_xlabel("Pulse height (ADC channel x 1000)", fontsize=fontsize)
    ax.set_ylabel("Counts (x 1000)", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.yaxis.set_major_formatter(lambda x, _: f"{x / 1000:.1f}")
    ax.tick_params(labelsize=fontsize)

### Plot Creation

In [ ]:
fig_folder = dl_folder / "figures"
fig_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
fig_height = 24
row_count = 3
row_ratios = [1, 1, 1]
row_total = sum(row_ratios)
row_heights = [(ratio / row_total) * fig_height for ratio in row_ratios]
row_1, row_2, row_3 = row_heights
height_a = row_1
height_b = row_1
height_d = row_2
height_e = row_3

fig_width = 24
col_ratios = [1, 1, 1, 1]
col_total = sum(col_ratios)
col_widths = [(ratio / col_total) * fig_width for ratio in col_ratios]
col_1, col_2, col_3, col_4 = col_widths
width_a = col_1 + col_2
width_b = col_3 + col_4
width_d = col_3 + col_4
width_e = col_2 + col_3

In [ ]:
fig, axs = plt.subplots(
    ncols=2,
    figsize=(width_a, height_a),
    dpi=600,
    # layout="constrained"
)
fig.subplots_adjust(wspace=0)
ax1, ax2 = axs
plot_figure_2a(ax1, ax2, fig)
fig.savefig(fig_folder / "si_fig_2a.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    figsize=(width_b, height_b),
    dpi=600,
    layout="constrained"
)
plot_figure_2b(ax)
fig.savefig(fig_folder / "si_fig_2b.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    figsize=(width_d, height_d),
    dpi=600,
    layout="constrained"
)
plot_figure_2d(ax)
fig.savefig(fig_folder / "si_fig_2d.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    figsize=(width_e, height_e),
    dpi=600,
    layout="constrained"
)
plot_figure_2e(ax)
fig.savefig(fig_folder / "si_fig_2e.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

## Base Data Export

In [ ]:
excel_folder = dl_folder / "excel"
excel_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_figure_2a_data():
    trace_df, phd_df, phd_curve_df, _ = get_figure_2a_data()
    trace_df.to_excel(
        excel_folder / "si_figure_2a_traces.xlsx",
    )
    phd_df.to_excel(
        excel_folder / "si_figure_2a_histogram.xlsx",
        header=["Bin midpoint (ADC channels)", "Bin width (ADC channels)", "Counts"]
    )
    phd_curve_df.to_excel(
        excel_folder / "si_figure_2a_curve.xlsx",
        header=["Counts"],
        index_label="Pulse height (ADC channels)"
    )


def export_figure_2b_data():
    fig_df, norm_factor_df, annot_positions_df = get_figure_2b_data()
    fig_df.to_excel(
        excel_folder / "si_figure_2b_histograms.xlsx",
    )
    norm_factor_df.to_excel(
        excel_folder / "si_figure_2b_norm_factors.xlsx",
    )
    annot_positions_df.to_excel(
        excel_folder / "si_figure_2b_annotation_positions.xlsx",
        index_label="Voltage (V)"
    )


def export_figure_2d_data():
    histo_df, gauss_df, fit_df = get_figure_2d_data()
    fit_df = fit_df.rename(
        index={"mu": "Mu", "sigma": "Sigma"},
        columns={"value": "Value"}
    )
    histo_df.to_excel(
        excel_folder / "si_figure_2d_histogram.xlsx",
        header=["Counts"],
        index_label="Bin midpoint (ADC channels)"
    )
    gauss_df.to_excel(
        excel_folder / "si_figure_2d_gaussian.xlsx",
        index_label="x"
    )
    fit_df.to_excel(
        excel_folder / "si_figure_2d_fit_parameters.xlsx",
    )


def export_figure_2e_data():
    df = get_figure_2e_data()
    df.to_excel(
        excel_folder / "si_figure_2e.xlsx",
        index_label="Pulse height (ADC channel)",
        header=["Background", "Neutron source"]
    )

In [ ]:
export_figure_2a_data()

In [ ]:
export_figure_2b_data()

In [ ]:
export_figure_2d_data()

In [ ]:
export_figure_2e_data()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()